# 3_10 Train regression MLP

Lean runnable training notebook (no slide deck).

Learn a curved 1D target with a small tanh MLP and **MSE**.
Same train / val / test discipline as the classifier lectures.

| Piece | Role |
|-------|------|
| data | noisy samples from a nonlinear `f(x)` |
| model | MLP `R -> R` |
| loss | `F.mse_loss` |
| loop | Adam + mini-batches; monitor val; report test once |


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(44)
np.random.seed(44)

N, NOISE = 240, 0.18
EPOCHS, LR, BATCH_SIZE = 100, 0.01, 32


def f_true(x):
    """Curved target -- needs a nonlinear MLP, not a single line."""
    return np.sin(1.7 * np.pi * x) + 0.35 * np.cos(3.3 * np.pi * x)


x = np.random.uniform(-1.0, 1.0, size=N).astype(np.float32)
y = (f_true(x) + NOISE * np.random.randn(N)).astype(np.float32)
perm = np.random.permutation(N)
x, y = x[perm], y[perm]

n_train, n_val = int(0.60 * N), int(0.20 * N)
x_tr, y_tr = x[:n_train], y[:n_train]
x_va, y_va = x[n_train:n_train + n_val], y[n_train:n_train + n_val]
x_te, y_te = x[n_train + n_val:], y[n_train + n_val:]

X_tr = torch.from_numpy(x_tr).unsqueeze(1)
Y_tr = torch.from_numpy(y_tr).unsqueeze(1)
X_va = torch.from_numpy(x_va).unsqueeze(1)
Y_va = torch.from_numpy(y_va).unsqueeze(1)
X_te = torch.from_numpy(x_te).unsqueeze(1)
Y_te = torch.from_numpy(y_te).unsqueeze(1)

print(f"train={len(X_tr)}  val={len(X_va)}  test={len(X_te)}")


## Model + loaders

Deeper tanh stack → more nonlinear capacity. Output is one continuous number (no Softmax).


In [ ]:
class MLPRegressor(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, 1),
        )

    def forward(self, x):     # x: (B, 1)
        return self.net(x)    # (B, 1)


def dramatic_init_(model: MLPRegressor) -> None:
    """Mildly wrong start -- visible at epoch 0, still easy to unlearn."""
    with torch.no_grad():
        last = model.net[-1]
        last.weight.mul_(0.3)
        last.weight.add_(1.2)
        last.bias.fill_(-0.8)


train_loader = DataLoader(TensorDataset(X_tr, Y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_va, Y_va), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te, Y_te), batch_size=BATCH_SIZE, shuffle=False)

torch.manual_seed(7)
model = MLPRegressor()
dramatic_init_(model)
opt = torch.optim.Adam(model.parameters(), lr=LR)
print(model)


## Training loop

```text
predict -> MSE -> zero_grad -> backward -> step
```

Log train / val each epoch. Touch **test** only at the end.


In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total, n = 0.0, 0
    for xb, yb in loader:
        loss = F.mse_loss(model(xb), yb)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / n


tr_hist, va_hist = [], []
for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va = run_epoch(val_loader, train=False)
    tr_hist.append(tr)
    va_hist.append(va)
    if epoch == 0 or (epoch + 1) % 20 == 0:
        print(f"epoch {epoch+1:03d}/{EPOCHS}  train_mse={tr:.4f}  val_mse={va:.4f}")


## Inference + one-shot test report


In [ ]:
model.eval()
with torch.no_grad():
    # Infer on a few test points
    yhat = model(X_te[:5])
    print("pred:", yhat.squeeze(1).numpy().round(3))
    print("true:", Y_te[:5].squeeze(1).numpy().round(3))

    test_mse = run_epoch(test_loader, train=False)
    test_mae = float(torch.mean(torch.abs(model(X_te) - Y_te)))
print(f"test MSE={test_mse:.4f}  MAE={test_mae:.4f}")
# Do not tune LR / epochs on this number.
